# 04 — Advanced Deep Architectures (CNNs, RNNs, Transformers) — Lab

This lab connects the lecture to a concrete computer-vision workflow. Students load a pre-trained ResNet-18, adapt the network head for CIFAR-10, train the model for a short transfer-learning run, and inspect feature maps from the first and last convolutional layers. The exercise stays within the lesson scope and does not use generative models or adversarial perturbations.

## Objectives

- Explain why convolution is parameter-efficient for spatial data and calculate a receptive field size for a given network depth.
- Extract and visualize intermediate feature maps from a convolutional layer and interpret the spatial patterns each map responds to.
- Distinguish attention-based from recurrence-based sequence modeling and state at least two scenarios where Transformers outperform RNNs.
- Fine-tune a pre-trained ResNet-18 on CIFAR-10 and visualize feature maps from the first and last convolutional layers.

## Prerequisites

You should already understand the lecture material from this lesson, including the 2D convolution operation, pooling, receptive fields, parameter sharing, recurrent memory, vanishing gradients, scaled dot-product attention, and the core Transformer encoder-decoder structure. You should also be comfortable with the previous module on deep learning fundamentals, especially forward propagation, loss functions, and optimization.

## Required Software and Packages

- Python 3.10+
- PyTorch
- torchvision
- matplotlib

No additional packages are required beyond the standard course environment.

## Environment Setup

Run the next cell to import the libraries needed for the lab and set up a device-aware training environment.

In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torchvision.models import resnet18
from torch.utils.data import DataLoader
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print("Setup cell executed successfully.")

## Background

Convolutional neural networks use shared kernels to scan the image and create feature maps at multiple scales. Early maps respond to simple geometry such as edges and corners, while deeper maps combine those cues into more complex structures. In this lab, a pre-trained ResNet-18 provides a strong visual backbone for CIFAR-10, and the final layer is replaced with a 10-class classifier.

The goal is not to train a network from scratch. The goal is to transfer the pretrained visual features to a new task and inspect how the representation changes from the first convolutional layer to the last one. This gives a direct view of the feature hierarchy that the lecture describes.

## Exercise Instructions

1. Load CIFAR-10, prepare the transforms, and replace the final linear layer of a pre-trained `resnet18` with a 10-class head. (3 min)
2. Fine-tune the model for 5 epochs on the training set using Adam with `lr=1e-4`, then evaluate the trained model on the test set. (4 min)
3. Register forward hooks on the first and last `Conv2d` blocks, run one test image through the model, and display the first 8 feature maps from each layer. (3 min)

In [ ]:
# Starter code: complete the TODOs below.

# TODO: define CIFAR-10 transforms
train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

# TODO: load CIFAR-10 and create train_loader and test_loader
train_dataset = datasets.CIFAR10(root=".", train=True, download=True, transform=train_transform)
test_dataset = datasets.CIFAR10(root=".", train=False, download=True, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# TODO: load a pretrained ResNet-18 and replace its output layer for 10 classes
model = resnet18(weights="IMAGENET1K_V1")
model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(device)

print(f"Train set size: {len(train_dataset)}")
print(f"Test set size: {len(test_dataset)}")
print(f"Output layer size: {model.fc.out_features}")

### Step 1: Prepare the model and data

In [ ]:
# Step 1: prepare the transfer-learning setup
# Replace the final layer, set the optimizer, and define a simple training loop.

model = resnet18(weights="IMAGENET1K_V1")
model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# Freeze the backbone, then unfreeze only after a brief sanity check.
for param in model.parameters():
    param.requires_grad = False
model.fc.requires_grad_(True)

# Train for a short run to confirm that the model can learn on CIFAR-10.
model.train()
example_images, example_labels = next(iter(train_loader))
example_images = example_images.to(device)
example_labels = example_labels.to(device)
logits = model(example_images)
loss = criterion(logits, example_labels)
optimizer.zero_grad()
loss.backward()
optimizer.step()

print(f"Sample batch logits shape: {tuple(logits.shape)}")
print(f"Sample batch loss: {loss.item():.4f}")

**Expected output:** A batch of logits with shape `(64, 10)` and a finite cross-entropy loss. This confirms the model accepts CIFAR-10 images and the new head produces one logit per class.

In [ ]:
# Step 2: fine-tune for 5 epochs and evaluate the model
# This code intentionally trains for a short transfer-learning pass and prints the final accuracy.

# Unfreeze the full network for transfer learning.
for param in model.parameters():
    param.requires_grad = True

epochs = 5
model.train()
train_losses = []

for epoch in range(epochs):
    running_loss = 0.0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)

    avg_loss = running_loss / len(train_dataset)
    train_losses.append(avg_loss)
    print(f"Epoch {epoch + 1:02d} | train loss: {avg_loss:.4f}")

model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        logits = model(images)
        predictions = logits.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

test_accuracy = correct / total
print(f"Test accuracy: {test_accuracy:.4f}")

**Expected output:** Five epoch-by-epoch loss values and a final test accuracy. A correctly configured transfer-learning run should produce a usable accuracy on CIFAR-10 after 5 epochs, even with a modest run on CPU.

In [ ]:
# Step 3: inspect the first and last convolutional feature maps
# Register forward hooks on the first and last Conv2d layers and visualize the first 8 maps.

model.eval()
feature_maps = {}

# Collect the first and last Conv2d blocks.
first_conv = model.conv1
last_conv = model.layer4[-1].conv2


def hook_fn(name):
    def _hook(module, inputs, output):
        feature_maps[name] = output.detach().cpu()
    return _hook

first_handle = first_conv.register_forward_hook(hook_fn("first_conv"))
last_handle = last_conv.register_forward_hook(hook_fn("last_conv"))

sample_image, _ = test_dataset[0]
sample_image = sample_image.unsqueeze(0).to(device)
with torch.no_grad():
    _ = model(sample_image)

first_handle.remove()
last_handle.remove()

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i in range(8):
    ax = axes[0, i]
    feature = feature_maps["first_conv"][0, i]
    ax.imshow(feature.numpy(), cmap="gray")
    ax.axis("off")
    if i == 0:
        ax.set_title("First conv")

    ax2 = axes[1, i]
    feature = feature_maps["last_conv"][0, i]
    ax2.imshow(feature.numpy(), cmap="gray")
    ax2.axis("off")
    if i == 0:
        ax2.set_title("Last conv")

plt.tight_layout()
plt.show()
print("Feature-map inspection completed.")

**Expected output:** A 2x8 grid of grayscale feature maps. The first convolutional layer should show sharp, localized edge-like patterns, while the last convolutional layer should show more abstract and spatially broader activations.

## Comprehension Questions

1. Why do convolutional layers use shared weights, and how does this reduce the parameter count relative to a fully-connected layer on a 32x32 image?
2. Why does a Transformer avoid the recurrence bottleneck that appears in RNNs and LSTMs when long-range dependencies span many positions?

In [ ]:
# Verification: confirm that the transfer-learning setup worked as expected.
assert isinstance(model, nn.Module), "The model should be a torch.nn.Module instance."
assert model.fc.out_features == 10, f"Expected 10 output classes, got {model.fc.out_features}."
assert len(train_loader) > 0, "The training loader should contain at least one batch."
assert isinstance(test_accuracy, float), "The evaluation step should compute a numeric accuracy value."
assert test_accuracy >= 0.70, f"Expected CIFAR-10 test accuracy >= 0.70, got {test_accuracy:.4f}."
assert "first_conv" in feature_maps and "last_conv" in feature_maps, "Feature maps were not captured from the requested hooks."
print("✓ PASS: The model was adapted to CIFAR-10, fine-tuned, and the feature maps were inspected successfully.")
print(f"  - Final CIFAR-10 test accuracy: {test_accuracy:.4f}")
print("  - The first and last convolutional blocks produced visible feature maps.")

## Optional Challenges

These challenges are clearly optional and are not required for the lab completion criteria.

1. Freeze the backbone for the first 2 epochs, then unfreeze the full network and continue training. Compare the final accuracy with the fully trainable version.
2. Replace the final layer with a smaller or larger hidden layer, then compare the feature maps and accuracy. Which design seems to preserve the most useful pre-trained features?

## Completion Criteria

You have successfully completed this lab when:

1. The model uses a pre-trained ResNet-18 with a 10-class output head for CIFAR-10.
2. The training loop runs for 5 epochs with Adam and prints the training loss each epoch.
3. The evaluation step computes a final CIFAR-10 test accuracy.
4. The verification cell prints a PASS result and confirms that the first and last convolutional blocks produced visible feature maps.
5. You can explain how parameter sharing and feature-map inspection connect to the CNN concepts in this lesson.